In [1]:
import os

import pandas as pd
import geopandas as gpd

import config as cfg

In [2]:
gdf = gpd.read_file(os.path.join(cfg.DATA_PATH, 'Results', 'region_2_results.geojson'))
vol_prod_df = pd.read_csv(os.path.join(cfg.DATA_PATH, 'Tabular Data', 'PSA-Fisheries - Total Value Production in Region 2.csv')).iloc[1:]
vol_prod_df.reset_index(inplace=True)

merged_df2 = vol_prod_df.merge(gdf[['ADM2_EN', 'Production Value']], left_on='Region 2 Provinces', right_on='ADM2_EN').drop(columns=['ADM2_EN', 'index'])
merged_df2 = merged_df2.rename(columns={'Production Value': '2022'})
merged_df2

,Region 2 Provinces,2015,2016,2017,2018,2019,2020,2022
0,Batanes,105298.31,89826.29,98987.49,103868.36,108494.61,122411.69,6.254191e+02
1,Cagayan,3882964.16,3724062.47,3747026.38,3623765.61,3588593.57,3685855.32,1.268641e+06
2,Isabela,871515.90,830036.66,860842.05,823595.47,812334.10,830228.27,3.822672e+05
3,Nueva Vizcaya,157893.21,162642.44,163907.97,158339.28,164469.02,180477.23,2.435740e+04
4,Quirino,71775.73,81179.62,86799.68,86079.48,87701.13,84428.46,4.669039e+03


In [4]:
gdf = gpd.read_file(os.path.join(cfg.DATA_PATH, 'Results', 'region_2_results.geojson'))
vol_prod_df = pd.read_csv(os.path.join(cfg.DATA_PATH, 'Tabular Data', 'PSA-Fisheries - Total Volume Production in Region 2.csv')).iloc[1:]
vol_prod_df.reset_index(inplace=True)

merged_df1 = vol_prod_df.merge(gdf[['ADM2_EN', 'Production Volume']], left_on='Region 2 Provinces', right_on='ADM2_EN').drop(columns=['ADM2_EN', 'index'])
merged_df1 = merged_df1.rename(columns={'Production Volume': '2022'})
merged_df1

,Region 2 Provinces,2015,2016,2017,2018,2019,2020,2022
0,Batanes,1010.08,813.06,880.13,878.97,889.16,856.80,4.377515
1,Cagayan,43171.62,40497.52,37853.65,35121.96,32815.10,31625.18,10885.128816
2,Isabela,9732.51,9484.04,9534.25,8959.58,9025.53,8765.47,4035.939789
3,Nueva Vizcaya,1685.60,1791.75,1834.81,1744.72,1790.23,1872.46,252.709229
4,Quirino,753.20,883.23,922.92,867.27,862.01,798.97,44.184419


In [9]:
def create_recommendation(row):
    if (row[r'2020-2022 % change Value'] < 0 and row[r'2020-2022 % change Volume'] < 0):
        row['Recommendation'] = 'Decrease Fishing Activity'
    elif (row[r'2020-2022 % change Value'] > 0 and row[r'2020-2022 % change Volume'] < 0):
        row['Recommendation'] = 'Decrease Fishing Acitivity (Keep Price)'
    elif (row[r'2020-2022 % change Value'] < 0 and row[r'2020-2022 % change Volume'] > 0):
        row['Recommendation'] = 'Increase Fishing Acitivity (Lower Price)'
    else:
        row['Recommendation'] = 'Increase Fishing Acitivity'
    return row

merged_df2[r'2020-2022 % change Value'] = (merged_df2['2022'] / merged_df2['2020']) - 1
merged_df1[r'2020-2022 % change Volume'] = (merged_df1['2022'] / merged_df1['2020']) - 1

merged_df = pd.concat([merged_df1[[r'2020-2022 % change Volume']], merged_df2[[r'2020-2022 % change Value']]], axis=1)

merged_df = merged_df.apply(create_recommendation, axis=1)

merged_df

,2020-2022 % change Volume,2020-2022 % change Value,Recommendation
0,-0.994891,-0.994891,Decrease Fishing Activity
1,-0.655808,-0.655808,Decrease Fishing Activity
2,-0.539564,-0.539564,Decrease Fishing Activity
3,-0.865039,-0.865039,Decrease Fishing Activity
4,-0.944698,-0.944698,Decrease Fishing Activity


In [11]:
merged_df.to_csv(os.path.join(cfg.DATA_PATH, 'Results', 'recommendations.csv'))